# JWST MAST imaging analysis notebook — improved version

This notebook is a more runnable, analysis-focused version of the original pedagogical JWST/MAST notebook. I treat “JSW” as **JWST** throughout.

## What changed

- Replaced brittle placeholders with a single configuration cell.
- Added safe `AUTO_DOWNLOAD` behavior so the notebook can be reviewed without immediately downloading large FITS files.
- Added robust MAST discovery and product-selection helpers.
- Removed hard-coded `example_i2d.fits`; the notebook now finds downloaded/local `*_i2d.fits*` products.
- Added FITS/HDU diagnostics before any science analysis.
- Added clearer mask, background, detection, deblending, photometry, calibration, QA, and export sections.
- Made multiband alignment, forced photometry, and external-catalog matching optional advanced sections.
- Added provenance capture so results can be traced back to query parameters, software versions, and FITS headers.

## Recommended workflow

1. Run the setup and configuration sections.
2. Run MAST discovery with `CONFIG["auto_download"] = False` first.
3. Inspect selected products.
4. Set `CONFIG["auto_download"] = True` only when the product list looks sensible, or place your own `*_i2d.fits` files in `mast_data/`.
5. Run the measurement and QA sections.
6. Export catalogues and provenance from the final section.


## 0. Setup

The core path uses `astroquery`, `astropy`, `photutils`, `reproject`, `pandas`, `numpy`, and `matplotlib`. `sep` and `scikit-learn` are optional comparison tools.

In [ ]:
# Run once in a fresh environment if packages are missing.
# Comment this out if your environment is already managed by conda/uv/poetry.
%pip install -q -U astroquery astropy photutils sep reproject pandas pyarrow scipy matplotlib tqdm scikit-learn


In [ ]:
from pathlib import Path
import json
import sys
import platform
import warnings
from datetime import datetime, timezone
import importlib.metadata as md

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from astropy.io import fits
from astropy.table import Table
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.wcs import WCS
from astropy.stats import SigmaClip
from astropy.convolution import convolve
from astropy.visualization import simple_norm, ImageNormalize, PercentileInterval, AsinhStretch

from astroquery.mast import Observations, Catalogs

from photutils.background import Background2D, MedianBackground
from photutils.segmentation import (
    make_2dgaussian_kernel,
    detect_threshold,
    detect_sources,
    deblend_sources,
    SourceCatalog,
)
from photutils.aperture import CircularAperture, CircularAnnulus, aperture_photometry

try:
    from reproject import reproject_interp
except Exception as exc:
    reproject_interp = None
    warnings.warn(f"reproject is unavailable: {exc}")

try:
    import sep
except Exception as exc:
    sep = None
    warnings.warn(f"sep is unavailable: {exc}")

print(f"Python: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")


## 1. Configuration

The notebook is target-agnostic, but it needs a default field for demonstration. The default below is a public JWST imaging field. Keep `auto_download=False` until you have inspected the product table.

In [ ]:
CONFIG = {
    # A known public JWST imaging field for demonstration. Replace with your science target.
    "target_name": "SMACS J0723.3-7327",
    "target_coord": "07h23m19.5s -73d27m15.6s",
    "search_radius": 2.5 * u.arcmin,

    # Archive constraints.
    "obs_collection": "JWST",
    "instruments": ("NIRCAM/IMAGE", "MIRI/IMAGE"),
    "dataproduct_type": "image",
    "public_only": True,
    "preferred_suffixes": ("_i2d.fits", "_i2d.fits.gz"),
    "max_observations_for_products": 20,
    "max_products_to_download": 4,

    # Keep False for safe review. Set True after inspecting selected_products.
    "auto_download": False,
    "data_dir": Path("mast_data"),
    "output_dir": Path("outputs"),

    # Image analysis controls.
    "dq_mask_mode": "do_not_use",   # "do_not_use" masks bit 1 only; "any" masks every nonzero DQ pixel.
    "background_box_size": (64, 64),
    "background_filter_size": (3, 3),
    "detection_nsigma": 2.5,
    "detection_npixels": 8,
    "kernel_fwhm_pix": 3.0,
    "kernel_size": 5,
    "deblend_nlevels": 32,
    "deblend_contrast": 0.001,

    # Aperture photometry controls.
    "aperture_radius_pix": 4.0,
    "annulus_radii_pix": (7.0, 11.0),
    "example_aperture_correction": 1.0,  # Replace with filter/PSF-specific correction for point-source science.

    # Optional external matching.
    "query_external_catalogs": False,
    "external_match_radius": 0.3 * u.arcsec,

    # Approximate local PSF width for a simple star/galaxy demo. Replace per filter.
    "psf_fwhm_pix": 2.8,
}

CONFIG["data_dir"].mkdir(exist_ok=True, parents=True)
CONFIG["output_dir"].mkdir(exist_ok=True, parents=True)
CONFIG


## 2. Environment and provenance manifest

Run this near the start so the analysis records which package versions created the result.

In [ ]:
PKGS = [
    "astroquery", "astropy", "photutils", "sep", "reproject",
    "pandas", "numpy", "scipy", "matplotlib", "pyarrow", "scikit-learn",
]

def package_version(name):
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return None

run_manifest = {
    "utc_started": datetime.now(timezone.utc).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "packages": {pkg: package_version(pkg) for pkg in PKGS},
    "config": {k: str(v) for k, v in CONFIG.items()},
}

manifest_path = CONFIG["output_dir"] / "environment_manifest.json"
manifest_path.write_text(json.dumps(run_manifest, indent=2))
print(f"Wrote {manifest_path}")
run_manifest["packages"]


## 3. MAST discovery

The helper below queries a target/region, then filters to public JWST imaging observations. It avoids assuming every MAST table has the same columns.

In [ ]:
def _string_col(table, name, default=""):
    if name not in table.colnames:
        return np.array([default] * len(table), dtype=str)
    return np.array([str(x).strip() for x in table[name]], dtype=str)


def query_public_jwst_imaging(config):
    coord = SkyCoord(config["target_coord"])
    obs = Observations.query_region(coord, radius=config["search_radius"])
    if len(obs) == 0:
        return obs

    mask = np.ones(len(obs), dtype=bool)
    mask &= _string_col(obs, "obs_collection").astype(str) == config["obs_collection"]

    if "dataproduct_type" in obs.colnames:
        mask &= np.char.upper(_string_col(obs, "dataproduct_type")) == config["dataproduct_type"].upper()

    if config["public_only"] and "dataRights" in obs.colnames:
        rights = np.char.upper(_string_col(obs, "dataRights"))
        mask &= np.isin(rights, ["PUBLIC", ""])

    if "instrument_name" in obs.colnames and config.get("instruments"):
        inst = _string_col(obs, "instrument_name")
        mask &= np.array([any(allowed in value for allowed in config["instruments"]) for value in inst])

    return obs[mask]


obs = query_public_jwst_imaging(CONFIG)
print(f"Matched observations: {len(obs)}")

summary_cols = [c for c in [
    "obs_id", "obsid", "proposal_id", "instrument_name", "filters",
    "dataproduct_type", "calib_level", "t_exptime", "dataRights", "s_ra", "s_dec"
] if c in obs.colnames]

obs[summary_cols][:10] if len(obs) else obs


## 4. Product listing, filtering, and optional download

The default science path uses stage-3 resampled imaging products ending in `_i2d.fits` or `_i2d.fits.gz`. These are the most convenient entry point for source detection, photometry, and multiband catalogues.

In [ ]:
def select_preferred_products(obs, config):
    if len(obs) == 0:
        return Table(), Table()

    n = min(len(obs), config["max_observations_for_products"])
    products = Observations.get_product_list(obs[:n])
    if len(products) == 0:
        return products, products

    # First use astroquery's product filter where possible.
    products = Observations.filter_products(
        products,
        productType=["SCIENCE"],
        extension=["fits", "fits.gz"],
    )

    filenames = _string_col(products, "productFilename")
    suffix_mask = np.array([
        any(fn.endswith(suffix) for suffix in config["preferred_suffixes"])
        for fn in filenames
    ])

    selected = products[suffix_mask]
    return products, selected


products, selected_products = select_preferred_products(obs, CONFIG)
print(f"All filtered FITS science products: {len(products)}")
print(f"Preferred stage-3 image products: {len(selected_products)}")

product_cols = [c for c in [
    "obsID", "obs_id", "productFilename", "productType", "description",
    "calib_level", "size", "dataURI"
] if c in selected_products.colnames]

selected_products[product_cols][:20] if len(selected_products) else selected_products


In [ ]:
downloaded_files = []

if CONFIG["auto_download"]:
    if len(selected_products) == 0:
        raise RuntimeError("No preferred products found. Broaden the query or inspect `products`.")

    n = min(len(selected_products), CONFIG["max_products_to_download"])
    manifest = Observations.download_products(
        selected_products[:n],
        download_dir=str(CONFIG["data_dir"]),
        flat=True,
    )
    print(manifest)
    if "Local Path" in manifest.colnames:
        downloaded_files = [Path(p) for p in manifest["Local Path"] if Path(p).exists()]
else:
    print("AUTO_DOWNLOAD is False. Put *_i2d.fits* files in mast_data/, or set CONFIG['auto_download'] = True and rerun this cell.")

local_files = sorted(CONFIG["data_dir"].glob("*_i2d.fits*"))
science_files = sorted(set(local_files + downloaded_files))
print(f"Science files available locally: {len(science_files)}")
for path in science_files[:10]:
    print(" -", path)


## 5. FITS inspection and image loading

Before doing any measurements, inspect the HDU structure and confirm which extensions are available. The analysis functions look for `SCI`, `ERR`, `DQ`, `VAR_*`, and `AREA` extensions when present.

In [ ]:
def fits_extension_summary(path):
    rows = []
    with fits.open(path, memmap=True) as hdul:
        for idx, hdu in enumerate(hdul):
            data = getattr(hdu, "data", None)
            rows.append({
                "idx": idx,
                "name": hdu.name,
                "class": hdu.__class__.__name__,
                "shape": None if data is None else data.shape,
                "dtype": None if data is None else str(data.dtype),
                "bunit": hdu.header.get("BUNIT", ""),
                "extver": hdu.header.get("EXTVER", ""),
            })
    return pd.DataFrame(rows)


if science_files:
    display(fits_extension_summary(science_files[0]))
else:
    print("No local science file yet. Continue through discovery/product inspection, then download or add FITS files.")


In [ ]:
def get_hdu_data(hdul, name, dtype="float32"):
    if name in hdul:
        arr = hdul[name].data
        if arr is None:
            return None
        return arr.astype(dtype) if dtype else arr
    return None


def load_jwst_image(path):
    path = Path(path)
    with fits.open(path, memmap=True) as hdul:
        if "SCI" not in hdul:
            raise ValueError(f"{path} has no SCI extension")

        sci = hdul["SCI"].data.astype("float32")
        sci_header = hdul["SCI"].header.copy()
        primary_header = hdul[0].header.copy()

        image = {
            "path": path,
            "sci": sci,
            "err": get_hdu_data(hdul, "ERR", "float32"),
            "dq": get_hdu_data(hdul, "DQ", None),
            "var_poisson": get_hdu_data(hdul, "VAR_POISSON", "float32"),
            "var_rnoise": get_hdu_data(hdul, "VAR_RNOISE", "float32"),
            "var_flat": get_hdu_data(hdul, "VAR_FLAT", "float32"),
            "area": get_hdu_data(hdul, "AREA", "float32"),
            "wcs": WCS(sci_header),
            "sci_header": sci_header,
            "primary_header": primary_header,
            "filter": sci_header.get("FILTER", primary_header.get("FILTER", "UNKNOWN")),
            "pupil": sci_header.get("PUPIL", primary_header.get("PUPIL", "")),
            "bunit": sci_header.get("BUNIT", primary_header.get("BUNIT", "")),
        }
    return image


def require_loaded_image():
    if "image" not in globals() or image is None:
        raise RuntimeError("No image is loaded. Add/download *_i2d.fits* files, rerun the file-discovery cell, then rerun this cell.")
    return image


image = load_jwst_image(science_files[0]) if science_files else None
if image is not None:
    print(f"Loaded: {image['path']}")
    print(f"Shape: {image['sci'].shape}; filter={image['filter']}; pupil={image['pupil']}; BUNIT={image['bunit']}")


## 6. Quicklook visualisation

Use robust percent/asinh scaling for JWST images. Do not use this display scaling for quantitative measurements; it is only for inspection.

In [ ]:
def show_image(data, title="", percentile=99.5, figsize=(8, 8), cmap="gray"):
    finite = np.isfinite(data)
    if not finite.any():
        raise ValueError("Image has no finite pixels")
    norm = simple_norm(data[finite], stretch="asinh", percent=percentile)
    plt.figure(figsize=figsize)
    plt.imshow(data, origin="lower", cmap=cmap, norm=norm)
    plt.colorbar(label="image units")
    plt.title(title)
    plt.xlabel("x [pixel]")
    plt.ylabel("y [pixel]")
    plt.show()


if image is not None:
    show_image(image["sci"], title=f"{image['path'].name} | {image['filter']} {image['pupil']} | {image['bunit']}")
else:
    print("No image loaded yet.")


## 7. Masks, background, and noise model

This is the step that turns a pretty-image workflow into an analysis workflow. The default DQ mask is deliberately conservative about only the `DO_NOT_USE` bit; switch to `dq_mask_mode="any"` if you want to mask all nonzero DQ values.

In [ ]:
def build_data_mask(data, dq=None, mode="do_not_use"):
    mask = ~np.isfinite(data)
    if dq is not None:
        if mode == "any":
            mask |= dq != 0
        elif mode == "do_not_use":
            # JWST DQ bit 1 is commonly DO_NOT_USE. This avoids masking every informational flag.
            mask |= (dq.astype(np.uint64) & 1) > 0
        else:
            raise ValueError("mode must be 'do_not_use' or 'any'")
    return mask


def safe_box_size(data_shape, requested):
    # Background2D box sizes should not exceed the image dimensions.
    return tuple(max(8, min(int(req), max(8, dim // 2))) for req, dim in zip(requested, data_shape))


image = require_loaded_image()
data = image["sci"]
mask = build_data_mask(data, image["dq"], mode=CONFIG["dq_mask_mode"])
box_size = safe_box_size(data.shape, CONFIG["background_box_size"])

sigma_clip = SigmaClip(sigma=3.0, maxiters=10)
bkg = Background2D(
    data,
    box_size=box_size,
    filter_size=CONFIG["background_filter_size"],
    mask=mask,
    sigma_clip=sigma_clip,
    bkg_estimator=MedianBackground(),
    exclude_percentile=50.0,
)

data_sub = data - bkg.background

if image["err"] is not None and image["err"].shape == data.shape:
    error_image = np.where(np.isfinite(image["err"]) & (image["err"] > 0), image["err"], bkg.background_rms)
else:
    error_image = bkg.background_rms

print(f"Masked fraction: {mask.mean():.3%}")
print(f"Background median: {np.nanmedian(bkg.background):.4g}")
print(f"Background RMS median: {np.nanmedian(bkg.background_rms):.4g}")

show_image(data_sub, title="Background-subtracted image", percentile=99.5)
show_image(bkg.background_rms, title="Background RMS map", percentile=99.0)


## 8. Source detection, deblending, and catalogue construction

Photutils is the primary path because it stays inside the Astropy table/WCS ecosystem. SEP is included later as a comparison branch.

In [ ]:
kernel = make_2dgaussian_kernel(
    fwhm=CONFIG["kernel_fwhm_pix"],
    size=CONFIG["kernel_size"],
)

conv = convolve(
    data_sub,
    kernel,
    mask=mask,
    normalize_kernel=True,
    nan_treatment="interpolate",
    preserve_nan=True,
)

threshold = detect_threshold(
    data_sub,
    nsigma=CONFIG["detection_nsigma"],
    background=0.0,
    error=error_image,
    mask=mask,
)

segm = detect_sources(
    conv,
    threshold,
    npixels=CONFIG["detection_npixels"],
    mask=mask,
)

if segm is None:
    raise RuntimeError("No sources detected. Try lowering detection_nsigma or inspecting the image/background.")

segm_deblend = deblend_sources(
    conv,
    segm,
    npixels=CONFIG["detection_npixels"],
    nlevels=CONFIG["deblend_nlevels"],
    contrast=CONFIG["deblend_contrast"],
    progress_bar=False,
)

cat = SourceCatalog(
    data_sub,
    segm_deblend,
    convolved_data=conv,
    error=error_image,
    background=bkg.background,
    wcs=image["wcs"],
)
cat_table = cat.to_table()

print(f"Detected sources before deblend: {segm.nlabels}")
print(f"Detected sources after deblend: {segm_deblend.nlabels}")
cat_table[:5]


In [ ]:
plt.figure(figsize=(8, 8))
plt.imshow(data_sub, origin="lower", cmap="gray", norm=simple_norm(data_sub, "asinh", percent=99.5))
plt.contour(segm_deblend.data > 0, levels=[0.5], linewidths=0.4)
plt.title("Deblended segmentation contours")
plt.xlabel("x [pixel]")
plt.ylabel("y [pixel]")
plt.show()


## 9. Aperture photometry and flux calibration

For JWST stage-3 imaging, many science images are in surface-brightness units such as `MJy/sr`. Aperture sums in that case must be multiplied by pixel solid angle before converting to Jy and AB magnitudes. For point-source science, apply a filter- and aperture-specific aperture correction rather than the placeholder value in `CONFIG`.

In [ ]:
def table_col_float(table, name):
    col = table[name]
    try:
        col = col.to_value()
    except Exception:
        pass
    return np.asarray(col, dtype=float)


x = table_col_float(cat_table, "xcentroid")
y = table_col_float(cat_table, "ycentroid")
positions = np.column_stack([x, y])

aper = CircularAperture(positions, r=CONFIG["aperture_radius_pix"])
ann = CircularAnnulus(
    positions,
    r_in=CONFIG["annulus_radii_pix"][0],
    r_out=CONFIG["annulus_radii_pix"][1],
)

aper_phot = aperture_photometry(data_sub, aper, error=error_image, mask=mask)
ann_phot = aperture_photometry(data_sub, ann, error=error_image, mask=mask)

local_bkg_per_pix = ann_phot["aperture_sum"] / ann.area
aper_net = aper_phot["aperture_sum"] - local_bkg_per_pix * aper.area

cat_table["aper_flux_image_units"] = aper_net
if "aperture_sum_err" in aper_phot.colnames:
    cat_table["aper_flux_err_image_units"] = aper_phot["aperture_sum_err"]

cat_table[:5]


In [ ]:
def pixar_sr_from_header_or_area(image):
    for hdr in [image["sci_header"], image["primary_header"]]:
        if "PIXAR_SR" in hdr:
            return float(hdr["PIXAR_SR"])
    if image["area"] is not None:
        return float(np.nanmedian(image["area"]))
    return None


def aperture_sum_to_jy(aperture_sum, image, aperture_correction=1.0):
    """Convert aperture sums to Jy when the image unit is known.

    For BUNIT='MJy/sr': sum has units MJy/sr * pixel, so multiply by pixel solid angle.
    For BUNIT='Jy' or 'Jy/pixel': sum is already Jy-like.
    """
    bunit = str(image["bunit"]).strip().lower()
    values = np.asarray(aperture_sum, dtype=float) * aperture_correction

    if "mjy" in bunit and "/sr" in bunit:
        pixar_sr = pixar_sr_from_header_or_area(image)
        if pixar_sr is None:
            raise ValueError("BUNIT looks like MJy/sr, but PIXAR_SR/AREA was not found.")
        return values * pixar_sr * 1e6

    if bunit in {"jy", "jy/pixel", "jy pix-1"} or "jy" == bunit:
        return values

    warnings.warn(f"Unknown or unsupported BUNIT='{image['bunit']}'. Returning NaNs for calibrated flux.")
    return np.full_like(values, np.nan, dtype=float)


flux_jy = aperture_sum_to_jy(
    cat_table["aper_flux_image_units"],
    image,
    aperture_correction=CONFIG["example_aperture_correction"],
)

mag_ab = np.full_like(flux_jy, np.nan, dtype=float)
good = np.isfinite(flux_jy) & (flux_jy > 0)
mag_ab[good] = -2.5 * np.log10(flux_jy[good] / 3631.0)

cat_table["aper_flux_jy"] = flux_jy
cat_table["aper_mag_ab"] = mag_ab

cat_table[["label", "xcentroid", "ycentroid", "aper_flux_jy", "aper_mag_ab"]][:10]


## 10. SEP comparison branch

This optional branch provides a fast Source-Extractor-like comparison. It is useful for teaching how detection choices affect catalogues.

In [ ]:
if sep is None:
    print("SEP is not installed; skipping this comparison branch.")
else:
    sep_data = np.ascontiguousarray(data_sub.astype(np.float32))
    sep_mask = np.ascontiguousarray(mask.astype(bool))

    sep_bkg = sep.Background(sep_data, mask=sep_mask)
    sep_sub = sep_data - sep_bkg.back()
    sep_objects, sep_segmap = sep.extract(
        sep_sub,
        thresh=CONFIG["detection_nsigma"],
        err=sep_bkg.rms(),
        mask=sep_mask,
        minarea=CONFIG["detection_npixels"],
        deblend_nthresh=CONFIG["deblend_nlevels"],
        deblend_cont=CONFIG["deblend_contrast"],
        segmentation_map=True,
    )
    sep_catalog = pd.DataFrame(sep_objects)
    print(f"SEP detected {len(sep_catalog)} sources")
    display(sep_catalog.head())


## 11. Simple star–galaxy feature engineering

This is not a publication-grade classifier. It is a transparent diagnostic plane based on source width relative to an assumed PSF and a crude concentration-like metric. Replace `psf_fwhm_pix` per filter/field, ideally from bright unsaturated stars or a PSF model.

In [ ]:
def optional_numeric_col(table, name, fill=np.nan):
    if name not in table.colnames:
        return np.full(len(table), fill, dtype=float)
    return table_col_float(table, name)


semi_a = optional_numeric_col(cat_table, "semimajor_sigma")
semi_b = optional_numeric_col(cat_table, "semiminor_sigma")
area = optional_numeric_col(cat_table, "area")
segment_flux = optional_numeric_col(cat_table, "segment_flux")
max_value = optional_numeric_col(cat_table, "max_value")

fwhm_eff = 2.355 * np.sqrt(semi_a * semi_b)
mean_segment_sb = segment_flux / np.maximum(area, 1)
concentration_proxy = max_value / np.where(mean_segment_sb > 0, mean_segment_sb, np.nan)
size_ratio = fwhm_eff / CONFIG["psf_fwhm_pix"]

class_rule = np.full(len(cat_table), "uncertain", dtype=object)
class_rule[(size_ratio < 1.20) & np.isfinite(concentration_proxy)] = "star_like"
class_rule[(size_ratio > 1.40) | (optional_numeric_col(cat_table, "ellipticity", fill=0) > 0.35)] = "galaxy_like"

cat_table["fwhm_eff_pix"] = fwhm_eff
cat_table["size_ratio_to_psf"] = size_ratio
cat_table["concentration_proxy"] = concentration_proxy
cat_table["class_rule"] = class_rule

pd.DataFrame({
    "label": np.asarray(cat_table["label"]),
    "aper_mag_ab": np.asarray(cat_table["aper_mag_ab"], dtype=float),
    "fwhm_eff_pix": fwhm_eff,
    "size_ratio_to_psf": size_ratio,
    "class_rule": class_rule,
}).head(10)


In [ ]:
plt.figure(figsize=(6, 4))
plt.scatter(cat_table["aper_mag_ab"], cat_table["fwhm_eff_pix"], s=10, alpha=0.6)
plt.axhline(CONFIG["psf_fwhm_pix"], linestyle="--", label="assumed PSF FWHM")
plt.gca().invert_xaxis()
plt.xlabel("Aperture AB magnitude")
plt.ylabel("Effective FWHM [pix]")
plt.title("Simple star/galaxy diagnostic")
plt.legend()
plt.show()


## 12. Quality assurance diagnostics

The catalogue is only as useful as its validation. At minimum, record masked fraction, background-RMS distribution, source counts, and blank-aperture noise.

In [ ]:
def random_blank_aperture_sums(data, mask, radius, n=300, seed=42):
    rng = np.random.default_rng(seed)
    ny, nx = data.shape
    xs = rng.uniform(radius + 1, nx - radius - 1, n * 4)
    ys = rng.uniform(radius + 1, ny - radius - 1, n * 4)
    xi = np.clip(xs.astype(int), 0, nx - 1)
    yi = np.clip(ys.astype(int), 0, ny - 1)
    good = ~mask[yi, xi]
    positions = np.column_stack([xs[good][:n], ys[good][:n]])
    if len(positions) == 0:
        return np.array([])
    apertures = CircularAperture(positions, r=radius)
    tbl = aperture_photometry(data, apertures, mask=mask)
    return np.asarray(tbl["aperture_sum"], dtype=float)

blank_sums = random_blank_aperture_sums(
    data_sub,
    mask,
    CONFIG["aperture_radius_pix"],
    n=300,
)

qa = {
    "file": str(image["path"]),
    "filter": image["filter"],
    "bunit": image["bunit"],
    "masked_fraction": float(mask.mean()),
    "n_sources_deblended": int(segm_deblend.nlabels),
    "background_median": float(np.nanmedian(bkg.background)),
    "background_rms_median": float(np.nanmedian(bkg.background_rms)),
    "background_rms_p16": float(np.nanpercentile(bkg.background_rms, 16)),
    "background_rms_p84": float(np.nanpercentile(bkg.background_rms, 84)),
    "blank_aperture_rms_image_units": float(np.nanstd(blank_sums)) if len(blank_sums) else np.nan,
}

qa_path = CONFIG["output_dir"] / "quality_assurance_summary.json"
qa_path.write_text(json.dumps(qa, indent=2))
print(f"Wrote {qa_path}")
pd.Series(qa)


In [ ]:
if len(blank_sums):
    plt.figure(figsize=(6, 4))
    plt.hist(blank_sums, bins=40, alpha=0.8)
    plt.xlabel("Blank-aperture sum [image units]")
    plt.ylabel("Count")
    plt.title("Empirical blank-aperture noise")
    plt.show()


## 13. Optional multiband alignment and forced photometry

This section reprojects additional `_i2d` images onto the first loaded image grid, then measures the same aperture positions in each band. This is a teaching-grade forced-photometry example; science-grade colour work should also handle PSF matching, correlated noise, and aperture corrections per filter.

In [ ]:
def guess_band_label(image):
    filt = str(image.get("filter", "UNKNOWN"))
    pupil = str(image.get("pupil", ""))
    return "_".join([x for x in [filt, pupil] if x and x != "UNKNOWN"]).lower() or "unknown"


def background_subtract_simple(arr, mask_arr, config):
    box = safe_box_size(arr.shape, config["background_box_size"])
    bb = Background2D(
        arr,
        box_size=box,
        filter_size=config["background_filter_size"],
        mask=mask_arr,
        sigma_clip=SigmaClip(sigma=3.0, maxiters=10),
        bkg_estimator=MedianBackground(),
        exclude_percentile=50.0,
    )
    return arr - bb.background, bb


multiband_tables = []

if len(science_files) < 2:
    print("Need at least two local science files for multiband forced photometry.")
elif reproject_interp is None:
    print("reproject is unavailable; skipping multiband alignment.")
else:
    ref_image = image
    ref_wcs = ref_image["wcs"]
    ref_shape = ref_image["sci"].shape
    forced_positions = positions
    forced_aper = CircularAperture(forced_positions, r=CONFIG["aperture_radius_pix"])

    for file_path in science_files[:CONFIG["max_products_to_download"]]:
        band_image = load_jwst_image(file_path)
        if Path(file_path) == ref_image["path"]:
            aligned = band_image["sci"]
        else:
            aligned, footprint = reproject_interp(
                (band_image["sci"], band_image["wcs"]),
                ref_wcs,
                shape_out=ref_shape,
            )

        band_mask = ~np.isfinite(aligned)
        band_sub, band_bkg = background_subtract_simple(aligned, band_mask, CONFIG)
        band_phot = aperture_photometry(band_sub, forced_aper, mask=band_mask)
        band_flux = np.asarray(band_phot["aperture_sum"], dtype=float)

        band_label = guess_band_label(band_image)
        multiband_tables.append(pd.DataFrame({
            "label": np.asarray(cat_table["label"]),
            f"flux_image_units_{band_label}": band_flux,
            f"file_{band_label}": str(file_path),
        }))

    multiband_df = multiband_tables[0]
    for tbl in multiband_tables[1:]:
        multiband_df = multiband_df.merge(tbl, on="label", how="outer")

    display(multiband_df.head())


## 14. Optional external catalogue matching

Gaia matching is useful for bright foreground stars and astrometric sanity checks. Pan-STARRS can add optical context where the field is covered. These calls require internet access and may take time, so they are disabled by default.

In [ ]:
if CONFIG["query_external_catalogs"]:
    from astroquery.gaia import Gaia

    src_coords = image["wcs"].pixel_to_world(x, y)
    centre = SkyCoord(CONFIG["target_coord"])

    gaia_job = Gaia.cone_search_async(centre, radius=CONFIG["search_radius"])
    gaia = gaia_job.get_results()
    gaia_coords = SkyCoord(gaia["ra"] * u.deg, gaia["dec"] * u.deg)

    idx, sep2d, _ = src_coords.match_to_catalog_sky(gaia_coords)
    matched = sep2d < CONFIG["external_match_radius"]

    cat_table["gaia_match_sep_arcsec"] = np.full(len(cat_table), np.nan)
    cat_table["gaia_match_sep_arcsec"][matched] = sep2d[matched].to_value(u.arcsec)

    if "SOURCE_ID" in gaia.colnames:
        cat_table["gaia_source_id"] = np.full(len(cat_table), "", dtype=object)
        cat_table["gaia_source_id"][matched] = np.asarray(gaia["SOURCE_ID"].astype(str))[idx[matched]]

    print(f"Gaia matches within {CONFIG['external_match_radius']}: {matched.sum()} / {len(cat_table)}")
else:
    print("External catalogue matching is disabled. Set CONFIG['query_external_catalogs']=True to enable it.")


## 15. Exports

Export both science products and provenance. ECSV preserves table metadata better than CSV; Parquet is convenient for larger downstream analysis; CSV is included for interoperability.

In [ ]:
output_catalog = cat_table.copy()

# Save catalogues.
ecsv_path = CONFIG["output_dir"] / "jwst_source_catalog.ecsv"
csv_path = CONFIG["output_dir"] / "jwst_source_catalog.csv"
parquet_path = CONFIG["output_dir"] / "jwst_source_catalog.parquet"

output_catalog.write(ecsv_path, overwrite=True)
output_catalog.to_pandas().to_csv(csv_path, index=False)

try:
    output_catalog.write(parquet_path, format="parquet", overwrite=True)
except Exception as exc:
    warnings.warn(f"Could not write Parquet file: {exc}")

# Save segmentation map with the science WCS/header.
seg_path = CONFIG["output_dir"] / "jwst_segmentation_map.fits"
seg_header = image["sci_header"].copy()
fits.writeto(seg_path, segm_deblend.data.astype(np.int32), header=seg_header, overwrite=True)

# Save a compact FITS-header provenance file.
header_provenance = {
    "science_file": str(image["path"]),
    "primary_header": {k: str(v) for k, v in image["primary_header"].items()},
    "sci_header": {k: str(v) for k, v in image["sci_header"].items()},
    "run_manifest": run_manifest,
    "qa": qa,
}
prov_path = CONFIG["output_dir"] / "analysis_provenance.json"
prov_path.write_text(json.dumps(header_provenance, indent=2))

print("Wrote:")
for p in [ecsv_path, csv_path, parquet_path, seg_path, prov_path]:
    print(" -", p)


## 16. Suggested next improvements

- Replace the example aperture correction with filter-specific encircled-energy or empirical curve-of-growth corrections.
- Estimate the PSF FWHM from bright, isolated stars in each image rather than using a fixed number.
- Add PSF matching before using aperture colours across filters with very different PSFs.
- Add injection/recovery simulations for completeness and false-positive rates.
- Add a negative-image test: run detection on `-data_sub` and compare false detections to positive detections.
- Add WCS residual checks after matching compact sources across filters.
- Use stage-2 `cal` products when teaching per-exposure artefacts; keep stage-3 `i2d` for the default catalogue path.

## References to consult while using the notebook

- MAST JWST primer and Astroquery MAST observation queries.
- JWST pipeline product suffix documentation, especially `rate`, `cal`, and `i2d`.
- Photutils documentation for `Background2D`, segmentation, `SourceCatalog`, and aperture photometry.
- Reproject documentation for common-grid resampling.
